In [1]:
from collections import Counter
from hashlib import sha256
from pathlib import Path
from zipfile import ZipFile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

In [2]:
SEED = 42

DATASET_URL = (
    "https://github.com/ultralytics/"
    "assets/releases/download/v0.0.0/"
    "crack-seg.zip"
)

DATASETS_ROOT = Path("/content/datasets")
ARCHIVE_PATH = DATASETS_ROOT / "crack-seg.zip"
DATASET_ROOT = DATASETS_ROOT / "crack-seg"

SEMANTIC_DATASET_ROOT =  DATASETS_ROOT / "crack-seg-semantic"
SEMANTIC_MASK_ROOT = SEMANTIC_DATASET_ROOT / "masks"

OUTPUT_ROOT = Path("/content/vision_unit_02_outputs/block_02")
FIGURE_ROOT = OUTPUT_ROOT / "figures"

In [3]:
for dir in [DATASETS_ROOT, SEMANTIC_MASK_ROOT, OUTPUT_ROOT, FIGURE_ROOT]:
    dir.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "val", "test")
EXPECTED_COUNTS = {
    "train": 3717,
    "val": 200,
    "test": 112,
}

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp",
}

print("Dataset root:", DATASET_ROOT)
print("Output root:", OUTPUT_ROOT)

Dataset root: /content/datasets/crack-seg
Output root: /content/vision_unit_02_outputs/block_02


**Download and extract**


In [6]:
dataset_ready = (DATASET_ROOT / "images" / "train").is_dir()

if not dataset_ready:
    if not ARCHIVE_PATH.is_file():
        print("Downloading Crack-Seg...")
        torch.hub.download_url_to_file(
            DATASET_URL, str(ARCHIVE_PATH), progress=True
        )
        
    print("Extracting dataset...")
    with ZipFile(ARCHIVE_PATH, "r") as zip_file:
        zip_file.extractall(DATASETS_ROOT)


if not DATASET_ROOT.is_dir():
    candidates = [
        path for path in DATASETS_ROOT.rglob("*")
        if path.is_dir() and (path / "images").is_dir() and "crack" in path.name.lower()
    ]

    if len(candidates) != 1:
        candidates = [
            path for path in DATASETS_ROOT.rglob("images")
            if path.is_dir()
        ]
        if len(candidates) == 1:
            candidates = [candidates[0].parent]

    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Could not locate Crack-Seg root in {DATASETS_ROOT}. "
            f"Found candidates: {candidates}"
        )

    DATASET_ROOT = candidates[0]

Extracting dataset...


In [7]:
for split in SPLITS:
    required_directories = [
        (DATASET_ROOT / "images" / split),
        (DATASET_ROOT / "labels" / split)
    ]
    
    for directory in (required_directories):
        if not directory.is_dir():
            raise FileExistsError(directory)

print("Dataset extracted:", DATASET_ROOT)

Dataset extracted: /content/datasets


### YOLO segmentation label format

**Collect files and pairing audit**


In [8]:
def collect_images(directory):
    return sorted(
        path for path in directory.rglob("*")
        if (path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    )

def relative_key(file_path, root):
    return (
        file_path.relative_to(root).with_suffix("").as_posix()
    )

In [9]:
count_row = []
pairing_issue_rows = []

for split in SPLITS:
    image_root = DATASET_ROOT / "images" / split
    label_root = DATASET_ROOT / "labels" / split
    
    image_paths = collect_images(image_root)
    label_paths = sorted(label_root.rglob("*.txt"))
    
    image_keys = {
        relative_key(path, image_root) for path in image_paths
    }
    label_keys = {
        relative_key(path, label_root) for path in label_paths
    }
    
    missing_labels = sorted(image_keys - label_keys)
    orphan_labels = sorted(label_keys - image_keys)
    
    for key in missing_labels:
        pairing_issue_rows.append({
            "split" : split,
            "key" : key,
            "issue" : "missing_label"
        })
        
    for key in orphan_labels:
        pairing_issue_rows.append({
            "split" : split,
            "key" : key,
            "issue" : "orphan_label"
        })
    
    count_row.append({
        "split" : split,
        "expected_images": EXPECTED_COUNTS[split],
        "actual_images" : len(image_paths),
        "label_files" : len(label_paths),
        "missing_labels" : len(missing_labels),
        "orphan_labels" : len(orphan_labels)
    })

In [10]:
count_dataframe = pd.DataFrame(count_row)

display(count_dataframe)
count_dataframe.to_csv(OUTPUT_ROOT / "dataset_counts.csv", index=False)

,split,expected_images,actual_images,label_files,missing_labels,orphan_labels
0,train,3717,3717,3717,0,0
1,val,200,200,200,0,0
2,test,112,112,112,0,0


**Polygon parser**


In [11]:
def normalized_polygon_area(points):
    x_coordinates = points[:, 0]
    y_coordinates = points[:, 1]
    
    forward = np.dot(x_coordinates, np.roll(y_coordinates, -1))
    backward = np.dot(y_coordinates, np.roll(x_coordinates, -1))
    
    return float(0.5 * abs(forward - backward))

In [14]:
def parse_yolo_segmentation_label(label_path):
    instances = []
    issues = []
    
    if not label_path.is_file():
        return isinstance, [{
            "line" : None,
            "issue" : "missing_labels",
            "deatils" : str(label_paths)
        }]
    
    raw_lines = label_path.read_text(encoding="utf-8").splitlines()
    nonempty_lines = [line.strip() for line in raw_lines if line.strip()]
    
    for line_number, line in enumerate(nonempty_lines, start=1):
        tokens = line.split()
        
        # class + minimum 3 (x,y) points
        if len(tokens) < 7:
            issues.append({
                "line" : line_number,
                "issue" : "too_few_values",
                "detail" : len(tokens)
            })
            continue
        
        coordinate_count = len(tokens) - 1
        
        if coordinate_count % 2 != 0:
            issues.append({
                "line": line_number,
                "issue": "odd_coordinate_count",
                "detail": coordinate_count
            })
            continue
        
        try:
            class_value = float(tokens[0])
            coordinates = np.array(tokens[1:], dtype=np.float32)
        
        except ValueError:
            issues.append({
                "line": line_number,
                "issue": "non_numeric_value",
                "detail": line[:100],
            })
            continue
        
        if not class_value.is_integer():
            issues.append({
                "line": line_number,
                "issue": "non_integer_class",
                "detail": class_value,
            })
            continue
        
        class_id = int(class_value)
        if class_id != 0:
            issues.append({
                "line": line_number,
                "issue": "unknown_class",
                "detail": class_id,
            })
            continue
        
        if not np.isfinite(coordinates).all():
            issues.append({
                "line": line_number,
                "issue": "non_finite_coordinate",
                "detail": "",
            })
            continue
        
        if not((coordinates >= 0.0).all() and (coordinates <= 1.0).all()):
            issues.append({
                "line": line_number,
                "issue": "coordinate_out_of_range",
                "detail" : (
                    f"min={coordinates.min()}, "
                    f"max={coordinates.max()}"
                ),
            })
            continue
        
        points = coordinates.reshape(-1, 2)
        unique_points = np.unique(points, axis=0)
        
        if len(unique_points) < 3:
            issues.append({
                "line": line_number,
                "issue": (
                    "fewer_than_3_"
                    "unique_points"
                ),
                "detail": len(unique_points)
            })
            continue
        
        polygon_area = normalized_polygon_area(points)
        if polygon_area <= 1e-10:
            issues.append({
                "line": line_number,
                "issue": "zero_area_polygon",
                "detail": polygon_area,
            })
            continue
        
        instances.append({
            "class_id" : class_id,
            "points": points,
            "normalized_area": polygon_area
        })
        
    return instances, issues

**Full annotation audit**


In [16]:
audit_rows = []
annotation_issue_rows = []

for split in SPLITS:
    image_root = DATASET_ROOT / "images" / split
    label_root = DATASET_ROOT / "labels" / split
    
    image_paths = collect_images(image_root)
    
    for image_path in tqdm(image_paths, desc=f"Auditing {split}"):
        relative_path = image_path.relative_to(image_root)
        label_path = (label_root / relative_path).with_suffix(".txt")
        
        image = cv2.imread(str(image_path))
        
        if image is None:
            annotation_issue_rows.append({
                "split": split,
                "image": str(relative_path),
                "label": str(label_path),
                "line": None,
                "issue": "unreadable_image",
                "detail": "",
            })
            continue
        
        height, width = image.shape[:2]
        instances, issues = parse_yolo_segmentation_label(label_path)
        
        for issue in issues:
            annotation_issue_rows.append({
                "split": split,
                "image": str(relative_path),
                "label": str(label_path),
                **issue,
            })
        
        total_vertices = sum(len(instance["points"]) for instance in instances)
        
        normalized_area_sum = sum(instance["normalized_area"] for instance in instances)
        
        audit_rows.append({
            "split": split,
            "relative_image": str(relative_path),
            "image_path": str(image_path),
            "label_path": str(label_path),
            "width": width,
            "height": height,
            "instances": len(instances),
            "total_vertices": (total_vertices),
            "polygon_area_sum": (normalized_area_sum),
            "empty_label": (len(instances) == 0)
        })

Auditing train:   0%|          | 0/3717 [00:00<?, ?it/s]

Auditing val:   0%|          | 0/200 [00:00<?, ?it/s]

Auditing test:   0%|          | 0/112 [00:00<?, ?it/s]

In [17]:
audit_dataframe = pd.DataFrame(audit_rows)

annotation_issues_dataframe = pd.DataFrame(annotation_issue_rows)

pairing_issues_dataframe = pd.DataFrame(pairing_issue_rows)


audit_dataframe.to_csv(
    OUTPUT_ROOT
    / "annotation_audit.csv",
    index=False,
)

annotation_issues_dataframe.to_csv(
    OUTPUT_ROOT
    / "annotation_issues.csv",
    index=False,
)

pairing_issues_dataframe.to_csv(
    OUTPUT_ROOT
    / "pairing_issues.csv",
    index=False,
)

In [18]:
audit_summary_dataframe = (
    audit_dataframe.groupby("split")
    .agg(
        images=("relative_image", "count"),
        total_instances=("instances", "sum"),
        mean_instances=("instances", "mean"),
        max_instances=("instances", "max"),
        empty_labels=("empty_label", "sum"),
        min_width=("width", "min"),
        max_width=("width", "max"),
        min_height=("height", "min"),
        max_height=("height", "max"),
    )
    .reset_index()
)

display(audit_summary_dataframe.round(4))

print("Pairing issues:", len(pairing_issues_dataframe))
print("Annotation issues:", len(annotation_issues_dataframe))

,split,images,total_instances,mean_instances,max_instances,empty_labels,min_width,max_width,min_height,max_height
0,test,112,148,1.3214,5,0,416,416,416,416
1,train,3717,4893,1.3164,18,0,416,416,416,416
2,val,200,249,1.2450,9,1,416,416,416,416


Pairing issues: 0
Annotation issues: 0
